# 1448. Count Good Nodes in Binary Tree
**Difficulty:** 🟡 Medium · **Topic:** Trees · **Pattern:** [DFS Basics](README.md) · **LeetCode:** https://leetcode.com/problems/count-good-nodes-in-binary-tree/

## 💡 Concepts

**Core concept(s):** **DFS (recursion)** that carries one extra value *down* the tree: the largest value seen on the path from the root so far.

**Why it applies here:** Whether a node is "good" depends only on the nodes **above it** (its root-to-node path). DFS walks exactly one root-to-node path at a time, so the path's maximum can ride along as a function argument.

**Key intuition:** A node is good if nothing above it is bigger, i.e. `node.val >= max of the path above`. Pass that max down; each node updates it and hands it to its children. Count good nodes on the way back up.

---

### 📚 What is a Binary Tree?
- A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**).
- The top is the **root**; childless nodes are **leaves**; **height** `h` is the longest root-to-leaf path.
- Every node has exactly one **path** back up to the root: that path is what this problem looks at.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
- **DFS** dives down one branch as far as possible, then backtracks and tries the next branch.
- It is usually written with **recursion**: a function that calls itself on each child.
- The **call stack** holds one frame per node on the current root-to-node path, so a function argument is a natural place to keep "what I know about the path so far".
- **Complexity:** each node is visited once → **O(n)** time; the call stack grows to the tree's height → **O(h)** space.
- **In Python:** a plain recursive function (the default recursion limit is 1000 frames, so very deep chains need `sys.setrecursionlimit`).

---

**Prerequisite knowledge:**
- A `TreeNode` class and building a tree from a LeetCode level-order list.
- Recursion with a base case.

## 📝 Problem

A node `X` is **good** if no node on the path from the root down to `X` has a value **greater than** `X.val`. Return the number of good nodes. (The root is always good.)

**Example 1**
```
        3            good: 3 (root), 4 (3 <= 4), 5 (max 4 <= 5), 3 (left-left: max 3 <= 3)
       / \           not good: 1 (3 above it is bigger)
      1   4
     /   / \         -> 4
    3   1   5
```
`root = [3,1,4,3,null,1,5]` → `4`

**Example 2:** `root = [3,3,null,4,2]` → `3` (the `2` sits under a `3`).

**Example 3:** `root = [1]` → `1`.

**Constraints:** `1 <= number of nodes <= 10^5`, `-10^4 <= Node.val <= 10^4`.

> Two approaches: re-scan the whole path at every node (`O(n·h)`), then carry the path's max down with DFS (`O(n)`). Only two are meaningfully distinct; an explicit-stack version of Approach 2 is the same idea with the same cost.

In [1]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_chain(n):
    """Right-leaning chain 1 -> 2 -> ... -> n (height n) — the benchmark's worst case."""
    root = None
    for v in range(n, 0, -1):              # build bottom-up so no recursion is needed
        root = TreeNode(v, None, root)
    return root

### Approach 1 — Brute Force: re-scan the path at every node

**Idea:**
1. Walk the tree with DFS, keeping the actual list of values on the current root-to-node `path`.
2. At each node, scan the whole `path`: the node is good if every value above it is `<= node.val`.
3. Push the node's value onto `path`, recurse into both children, then pop it off (backtrack).

**Time complexity:** `O(n·h)` — each of the `n` nodes scans a path of up to `h` values (`O(n log n)` balanced, `O(n²)` for a chain).

**Space complexity:** `O(h)` — the `path` list and the call stack both hold one entry per level.

In [2]:
def good_nodes_brute(root: Optional[TreeNode]) -> int:
    path = []                              # values from the root down to the current node's parent
    def dfs(node):
        if not node:
            return 0                       # empty subtree has no good nodes
        # Good if nothing above is bigger: scan every ancestor's value (the slow part).
        good = 1 if all(p <= node.val for p in path) else 0
        path.append(node.val)              # this node is now an ancestor of its children
        good += dfs(node.left) + dfs(node.right)
        path.pop()                         # backtrack: leave this branch
        return good
    return dfs(root)

### Approach 2 — DFS carrying the path max (optimal)

**Idea:** We never need the whole path, only its **largest** value.
1. `dfs(node, max_so_far)` receives the biggest value on the path above `node`.
2. **Base case:** an empty node contributes `0`.
3. **Work at this node:** it is good if `node.val >= max_so_far`; the new max for the children is `max(max_so_far, node.val)`.
4. **Combine:** `good + dfs(left, new_max) + dfs(right, new_max)` goes back up to the parent.
5. Start with `max_so_far = root.val` so the root counts as good.

**Time complexity:** `O(n)` — every node does `O(1)` work once.

**Space complexity:** `O(h)` — the recursion stack, one frame per level (`O(log n)` balanced, `O(n)` for a chain).

In [3]:
def good_nodes_dfs(root: Optional[TreeNode]) -> int:
    def dfs(node, max_so_far):
        if not node:
            return 0                                   # base case: empty adds nothing
        good = 1 if node.val >= max_so_far else 0      # nothing above is bigger -> good
        new_max = max(max_so_far, node.val)            # state handed DOWN to the children
        # Combine: this node's 0/1 plus the counts coming UP from both subtrees.
        return good + dfs(node.left, new_max) + dfs(node.right, new_max)
    return dfs(root, root.val) if root else 0          # root is good: compare it to itself

In [4]:
# Correctness check
tests = [
    ([3,1,4,3,None,1,5], 4),               # LeetCode example 1
    ([3,3,None,4,2], 3),                   # LeetCode example 2: equal values count as good
    ([1], 1),                              # single node: the root is always good
    ([5,4,3,2,1], 1),                      # every child is smaller -> only the root
    ([-1,-2,0], 2),                        # negatives: root -1 and the 0 are good
    ([2,None,4,10,8,None,None,4], 4),      # 4 under 8 is NOT good (8 is above it)
]
for vals, exp in tests:
    root = build_tree(vals)
    a, b = good_nodes_brute(root), good_nodes_dfs(root)
    print(f"{vals} -> brute={a}, dfs={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

[3, 1, 4, 3, None, 1, 5] -> brute=4, dfs=4 | expected=4
[3, 3, None, 4, 2] -> brute=3, dfs=3 | expected=3
[1] -> brute=1, dfs=1 | expected=1
[5, 4, 3, 2, 1] -> brute=1, dfs=1 | expected=1
[-1, -2, 0] -> brute=2, dfs=2 | expected=2
[2, None, 4, 10, 8, None, None, 4] -> brute=4, dfs=4 | expected=4

All tests passed


## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

The worst case for the brute force is a **chain** `1 → 2 → … → n` (height `n`): every node is good, so `all(...)` never stops early and scans the full path. Brute should show ≈4× (`O(n·h) = O(n²)`), the DFS ≈2× (`O(n)`). The chain is `n` levels deep, so we raise the recursion limit for this cell.

In [5]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

sys.setrecursionlimit(20000)                # the chain is n levels deep

def make_worst_case(n):
    return (build_chain(n),)               # increasing chain: every node good, full path scans
solutions = {
    "brute O(n*h)": good_nodes_brute,
    "dfs   O(n)":   good_nodes_dfs,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, repeats=3, plot=True)   # best of 3 runs cuts timer noise


brute O(n*h)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       9.21 |           n/a
     2000 |      29.89 |         3.25x


     4000 |     116.42 |         3.90x


     8000 |     461.23 |         3.96x

dfs   O(n)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       0.09 |           n/a
     2000 |       0.21 |         2.25x
     4000 |       0.41 |         1.91x
     8000 |       0.82 |         2.02x

Ratios per doubling: ~1x=>constant/log, ~2x=>linear or n log n, ~4x=>quadratic, ~8x=>cubic.

[plot skipped] matplotlib not installed - run: pip install matplotlib


{'brute O(n*h)': [9.20654193032533,
  29.886542004533112,
  116.42404203303158,
  461.22595807537436],
 'dfs   O(n)': [0.09437499102205038,
  0.21237507462501526,
  0.40625000838190317,
  0.8218339644372463]}

## 🧩 Patterns Learned

- **Pattern:** [DFS Basics](README.md) — the template plus one argument passed *down* (`max_so_far`); the count is combined on the way *up*.
- **State down, answer up:** facts about the ancestors (max, min, bounds, path sum) travel as arguments; facts about the subtree (counts, heights) travel back as return values.
- **Keep a summary, not the path:** when only the path's max matters, carry one number instead of the whole list — that turns `O(n·h)` into `O(n)`.
- **Signal:** "on the path from the root to X", "no ancestor greater than", "compared with every node above it".
- **Related problems:** Validate Binary Search Tree (pass `(low, high)` down), Path Sum (pass the remaining sum down), Maximum Depth of Binary Tree (answer up only).
- **Common pitfalls:** (1) using `>` instead of `>=` — equal values are good; (2) starting `max_so_far` at `0` breaks negative values (start at `root.val` or `-inf`); (3) updating the max before checking the node; (4) Python's recursion limit on very deep (chain) trees.